# Lab 1: Build a Tool-Using Agent

You will build an **enterprise service-request assistant**: an agent that
classifies a request, checks a simulated service-status table, searches a knowledge file and proposes a
next step, recording an action only after human approval.

**How this works.** Almost all the code is written for you. Seven short gaps, **CODE-ALONG 1 to 7**, are
left to complete: one to three lines each. Every gap has a comment
right above it saying what to write, why, a hint and how to check it. More hints, one level at a time,
are in `docs/HINTS.md`.

1. Open the file from the **Files** panel on the left (double-click), find the gap (Ctrl+F `CODE-ALONG`).
2. Replace the `raise NotImplementedError(...)` line with your code, keeping the indentation.
3. Save with **Ctrl+S**, then run the check cell below it.

Before every agent run, write down what you expect it to do, then check the trace.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Your Gemini key must be in Colab
Secrets (key icon on the left) as `GEMINI_API_KEY`, with notebook access switched on.

## Start here: connect your Drive and run setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/lab1_service_agent'   # change this if you saved the folder elsewhere
%cd {PROJECT}

In [ ]:
# Installs packages, starts Ollama in this runtime and downloads the open model (a few minutes the first time).
%run scripts/colab_setup.py

## Check the setup

Every item should say PASS. Fix any FAIL before going on.

In [ ]:
%run scripts/00_check_setup.py

## Data with pandas and NumPy, and the GPU check

1. Run the data step and read the output. Which row has no status?
2. Open `scripts/01_explore_data.py`, change the two settings marked **EDIT**, predict what will change, rerun.
3. Run the GPU check. **Predict first:** at `--size 2048`, will the GPU or the CPU be faster? Then try 256 and 4096.

In [ ]:
%run scripts/01_explore_data.py

In [ ]:
%run scripts/gpu_check.py --size 2048

## The three tools: complete the gaps

Open **`src/tools.py`** and complete the five gaps in order. After each one, save and run the check
cell: the number of OK checks goes up by one or two each time.

| Gap | Tool | What it adds |
| --- | --- | --- |
| CODE-ALONG 1 | `read_status` | an unknown service returns an error instead of a made-up status |
| CODE-ALONG 2 | `read_status` | an empty status cell becomes `"unknown"`, never a guess |
| CODE-ALONG 3 | `read_status` | NumPy mean and maximum of the sensor readings |
| CODE-ALONG 4 | `search_knowledge` | a score for how well each note matches the query |
| CODE-ALONG 5 | `record_proposed_action` | no approval, no action |

In [ ]:
# Run after EVERY gap (CODE-ALONG 1 to 5). It tests the tools directly, without a model.
%run scripts/try_tools.py

**Guided change.** In `src/tools.py`, find `EXTRA_ALIASES` near the top and add one line of your own,
for example `"teams call": "video",`. Then look it up:

In [ ]:
%run scripts/try_tools.py --service "teams call"

**Connect the open model.** When `try_tools.py` shows 7 of 7, run the agent on R01.

**Your prediction** (edit this cell before running): which tool will the agent call, with which argument,
and what will the final answer say? *...*

In [ ]:
%run scripts/03_run_agent.py --provider open --request R01 --show-trace

In the trace, find the **tool call**, its **arguments** and the **result**. Which events are *reason*,
*act*, *observe* and *learn*? Every trace is saved in `outputs/traces/`.

**Plain model versus agent.** The same request, answered first by the model on its
own (no tools, no state, no checks), then by the agent. What does each answer know about the real status,
and where did that knowledge come from?

In [ ]:
%run scripts/02_plain_vs_agent.py --provider open --request R01

## Debugging: an invalid argument and a missing status

- **R03** names a service that does not exist. What does your tool return, and does the agent invent anything?
- **R04** asks about a service whose status cell is empty. Does the agent say "unknown" or guess?

In [ ]:
%run scripts/03_run_agent.py --provider open --request R03 R04 --show-trace

## Run the same benchmark through Gemini

The same 8 requests, tools and prompt go through the open model and Gemini. The free Gemini tier is
rate-limited, so this takes a few minutes. Afterwards, open `outputs/comparison_summary.md` and
`outputs/comparison.csv`, then write **one behavioural difference** in `docs/architecture.md`
(section 5, Model comparison note).

*No GPU?* Use `--providers gemini_openai gemini` instead.

In [ ]:
%run scripts/04_compare_models.py --providers open gemini

## Safeguards: see the problem, switch them on, complete the gaps

**1. Without safeguards.** Run this first and read what goes wrong.

In [ ]:
%run scripts/try_safeguards.py

**2. Switch them on.** In **`src/config.py`**, change `False` to `True` for `STEP_LIMIT_ON`,
`RESULT_VALIDATION_ON` and `APPROVAL_GATE_ON`. Save.

**3. Complete the gaps.** In **`src/safeguards.py`**, write **CODE-ALONG 6** (step limit, one line) and
**CODE-ALONG 7** (approve only on a clear yes, one line). Safeguard 2 is already written: read what it checks.

**4. Test what each one stops.** Compare with step 1.

In [ ]:
%run scripts/try_safeguards.py

**5. Try it with the open model.** Run R02 and answer the approval prompt yourself: once `y`, once `n`.
What changes in `outputs/action_log.csv` and in the trace? Then run R01 with a step limit of 1.

In [ ]:
%run scripts/03_run_agent.py --provider open --request R02 --approval interactive

In [ ]:
%run scripts/03_run_agent.py --provider open --request R01 --max-steps 1 --approval approve

## Acceptance tests and the architecture blueprint

Run the four acceptance cases (target: **at least 3 of 4**). `--extra` also runs the missing-status case.
Then complete every **TODO** in **`docs/architecture.md`**: the capstone use-case brief, the
observe-reason-act-learn mapping, the tool contracts and your metrics.

In [ ]:
%run scripts/05_run_tests.py --provider open --extra

## Save your evidence

In [ ]:
%run scripts/06_evidence_check.py

---
## Optional extension: write a fourth tool

After the main tasks, open **`src/extension.py`** and write `sensor_trend` from the step-by-step hints in the
file (about 10 lines). Check it, then set `EXTENSION_TOOL_ON = True` in `src/config.py` and check that the
agent uses it on R08.

In [ ]:
%run scripts/try_tools.py --extension

In [ ]:
%run scripts/03_run_agent.py --provider open --request R08 --show-trace

---
## After the lab

1. Change one tool (for example, add a field to `read_status` or a new alias), rerun `try_tools.py` and
   the tests, and explain the change in `docs/architecture.md`.
2. Run both providers on two new requests of your own:
   `%run scripts/03_run_agent.py --provider gemini --text "your request"`.
3. Finish the Project 1 blueprint with your success metrics.
4. If no GPU was available, run `scripts/gpu_check.py` on a GPU runtime.

## If something goes wrong

| Problem | Fix |
| --- | --- |
| `NOT YET -> CODE-ALONG n` | That gap is next. Read the comment above it, or `docs/HINTS.md`. |
| `IndentationError` | Your line must start in the same column as the `raise` line you replaced. |
| Your edit has no effect | Save the file (Ctrl+S) before rerunning the cell. |
| Colab disconnected | Rerun the two cells under *Start here*. Your files are safe in Drive. |
| `Ollama server: FAIL` | Rerun `%run scripts/colab_setup.py`. |
| No GPU available | Use `--provider gemini_openai` for the agent and do the GPU check later. |
| Gemini rate limit (429) | Wait a minute; the code retries once automatically. |
| No model at all | `--provider mock` runs a rule-based stand-in so you can still test tools and safeguards. |